In [ ]:

# Evaluation of Energy-Efficient Global Architecture Search on CIFAR-10

!pip -q install thop pandas

import os
import copy
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from thop import profile


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

-
BATCH_SIZE = 64
EPOCHS = 20
LR = 0.001
NUM_CLASSES = 10
IMAGE_SIZE = 32

# Energy model coefficient
# Energy (J) = alpha * FLOPs * num_epochs
ENERGY_ALPHA = 2.0e-9


transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_full = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

# Split training into train/val
train_size = 45000
val_size = 5000
train_set, val_set = random_split(train_full, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


SEARCH_SPACE = {
    "channels": [16, 32, 64],
    "depth": [2, 3, 4],
    "kernel_size": [3, 5],
}

def random_architecture():
    return {
        "channels": random.choice(SEARCH_SPACE["channels"]),
        "depth": random.choice(SEARCH_SPACE["depth"]),
        "kernel_size": random.choice(SEARCH_SPACE["kernel_size"]),
    }

def mutate_architecture(arch, p=0.7):
    new_arch = copy.deepcopy(arch)
    for key in SEARCH_SPACE:
        if random.random() < p:
            new_arch[key] = random.choice(SEARCH_SPACE[key])
    return new_arch

def arch_to_str(arch):
    return f"channels={arch['channels']}, depth={arch['depth']}, k={arch['kernel_size']}"

class SimpleCNN(nn.Module):
    def __init__(self, arch, num_classes=10):
        super().__init__()
        channels = arch["channels"]
        depth = arch["depth"]
        kernel_size = arch["kernel_size"]
        padding = kernel_size // 2

        layers = []
        in_ch = 3

        for i in range(depth):
            out_ch = channels
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding))
            layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            if i < depth - 1:
                layers.append(nn.MaxPool2d(2))
            in_ch = out_ch

        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def train_one_model(model, train_loader, val_loader, epochs=5, lr=0.001):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = outputs.max(1)
            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

        train_acc = 100.0 * correct / total
        val_acc = evaluate_accuracy(model, val_loader)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch [{epoch+1}/{epochs}] | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    model.load_state_dict(best_state)
    return model, best_val_acc

@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()
    return 100.0 * correct / total

def compute_flops_and_params(model):
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    return flops, params

def estimate_energy_joules(flops, epochs, alpha=ENERGY_ALPHA):
    return alpha * flops * epochs


def evaluate_candidate_architecture(arch):
    print("\nEvaluating architecture:", arch_to_str(arch))
    model = SimpleCNN(arch, num_classes=NUM_CLASSES)
    model, val_acc = train_one_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
    test_acc = evaluate_accuracy(model, test_loader)
    flops, params = compute_flops_and_params(model)
    energy_j = estimate_energy_joules(flops, EPOCHS)

    result = {
        "Architecture": arch_to_str(arch),
        "Val_Accuracy (%)": round(val_acc, 2),
        "Test_Accuracy (%)": round(test_acc, 2),
        "Energy (J)": round(energy_j, 4),
        "FLOPs (M)": round(flops / 1e6, 2),
        "Params": int(params),
    }
    return result, model


largest_backbone = {
    "channels": max(SEARCH_SPACE["channels"]),
    "depth": max(SEARCH_SPACE["depth"]),
    "kernel_size": max(SEARCH_SPACE["kernel_size"]),
}

smallest_backbone = {
    "channels": min(SEARCH_SPACE["channels"]),
    "depth": min(SEARCH_SPACE["depth"]),
    "kernel_size": min(SEARCH_SPACE["kernel_size"]),
}

# Accuracy-only search:
# train all candidates, pick highest validation accuracy
def accuracy_only_search(candidates):
    best_result = None
    best_model = None
    best_val = -1

    for arch in candidates:
        result, model = evaluate_candidate_architecture(arch)
        if result["Val_Accuracy (%)"] > best_val:
            best_val = result["Val_Accuracy (%)"]
            best_result = result
            best_model = model

    return best_result, best_model

# Random search:
# randomly sample one architecture
def random_search(candidates):
    arch = random.choice(candidates)
    return evaluate_candidate_architecture(arch)

# OrchNAS:
# score = val_acc - lambda * energy
def orchnas_search(candidates, lambda_energy=5.0):
    best_score = -1e18
    best_result = None
    best_model = None

    for arch in candidates:
        result, model = evaluate_candidate_architecture(arch)
        score = result["Val_Accuracy (%)"] - lambda_energy * result["Energy (J)"]
        print(f"OrchNAS score = {score:.4f}")

        if score > best_score:
            best_score = score
            best_result = result
            best_model = model

    return best_result, best_model


candidate_pool = []

# include fixed extremes
candidate_pool.append(largest_backbone)
candidate_pool.append(smallest_backbone)

# add random + mutated architectures
base_arch = random_architecture()
for _ in range(6):
    candidate_pool.append(mutate_architecture(base_arch, p=0.8))

# remove duplicates
unique_pool = []
seen = set()
for arch in candidate_pool:
    key = (arch["channels"], arch["depth"], arch["kernel_size"])
    if key not in seen:
        seen.add(key)
        unique_pool.append(arch)

candidate_pool = unique_pool

print("\nCandidate Pool:")
for i, arch in enumerate(candidate_pool):
    print(f"{i+1}. {arch_to_str(arch)}")


all_results = []

start_time = time.time()

print("\n==============================")
print("Running Largest Backbone")
print("==============================")
largest_result, _ = evaluate_candidate_architecture(largest_backbone)
largest_result["Method"] = "Largest Backbone"
all_results.append(largest_result)

print("\n==============================")
print("Running Smallest Backbone")
print("==============================")
smallest_result, _ = evaluate_candidate_architecture(smallest_backbone)
smallest_result["Method"] = "Smallest Backbone"
all_results.append(smallest_result)

print("\n==============================")
print("Running Accuracy-Only Search")
print("==============================")
acc_only_result, _ = accuracy_only_search(candidate_pool)
acc_only_result["Method"] = "Accuracy-Only Search"
all_results.append(acc_only_result)

print("\n==============================")
print("Running Random Search")
print("==============================")
random_result, _ = random_search(candidate_pool)
random_result["Method"] = "Random Search"
all_results.append(random_result)

print("\n==============================")
print("Running OrchNAS")
print("==============================")
orchnas_result, _ = orchnas_search(candidate_pool, lambda_energy=5.0)
orchnas_result["Method"] = "OrchNAS"
all_results.append(orchnas_result)

end_time = time.time()


results_df = pd.DataFrame(all_results)
results_df = results_df[[
    "Method",
    "Architecture",
    "Val_Accuracy (%)",
    "Test_Accuracy (%)",
    "Energy (J)",
    "FLOPs (M)",
    "Params"
]]

csv_path = "orchnas_global_architecture_search_results.csv"
results_df.to_csv(csv_path, index=False)

# ----------------------------
# 12. Print Results Clearly
# ----------------------------
print("\n================ FINAL RESULTS ================")
print(results_df.to_string(index=False))

print(f"\nResults saved to: {csv_path}")


